In [1]:
import pandas as pd

In [2]:
import requests 

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [3]:
documents[2]

{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
 'section': 'General course-related questions',
 'question': 'Course - Can I still join the course after the start date?',
 'course': 'data-engineering-zoomcamp'}

In [4]:
df = pd.DataFrame(documents, columns=['course', 'section', 'question', 'text'])
df.head()

,course,section,question,text
0,data-engineering-zoomcamp,General course-related questions,Course - When will the course start?,The purpose of this document is to capture fre...
1,data-engineering-zoomcamp,General course-related questions,Course - What are the prerequisites for this c...,GitHub - DataTalksClub data-engineering-zoomca...
2,data-engineering-zoomcamp,General course-related questions,Course - Can I still join the course after the...,"Yes, even if you don't register, you're still ..."
3,data-engineering-zoomcamp,General course-related questions,Course - I have registered for the Data Engine...,You don't need it. You're accepted. You can al...
4,data-engineering-zoomcamp,General course-related questions,Course - What can I do before the course starts?,You can start by installing and setting up all...


In [5]:
df.tail()

,course,section,question,text
943,mlops-zoomcamp,Module 6: Best practices,Github actions: Permission denied error when e...,Problem description\nThis is the step in the c...
944,mlops-zoomcamp,Module 6: Best practices,Managing Multiple Docker Containers with docke...,Problem description\nWhen a docker-compose fil...
945,mlops-zoomcamp,Module 6: Best practices,AWS regions need to match docker-compose,Problem description\nIf you are having problem...
946,mlops-zoomcamp,Module 6: Best practices,Isort Pre-commit,Problem description\nPre-commit command was fa...
947,mlops-zoomcamp,Module 6: Best practices,How to destroy infrastructure created via GitH...,Problem description\nInfrastructure created in...


In [6]:
# Filtering with pandas
df[df.course == "data-engineering-zoomcamp"]

,course,section,question,text
0,data-engineering-zoomcamp,General course-related questions,Course - When will the course start?,The purpose of this document is to capture fre...
1,data-engineering-zoomcamp,General course-related questions,Course - What are the prerequisites for this c...,GitHub - DataTalksClub data-engineering-zoomca...
2,data-engineering-zoomcamp,General course-related questions,Course - Can I still join the course after the...,"Yes, even if you don't register, you're still ..."
3,data-engineering-zoomcamp,General course-related questions,Course - I have registered for the Data Engine...,You don't need it. You're accepted. You can al...
4,data-engineering-zoomcamp,General course-related questions,Course - What can I do before the course starts?,You can start by installing and setting up all...
...,...,...,...,...
430,data-engineering-zoomcamp,Workshop 2 - RisingWave,Unable to Open Dashboard as xdg-open doesn’t o...,Refer to the solution given in the first solut...
431,data-engineering-zoomcamp,Workshop 2 - RisingWave,Resolving Python Interpreter Path Inconsistenc...,Example Error:\nWhen attempting to execute a P...
432,data-engineering-zoomcamp,Workshop 2 - RisingWave,How does windowing work in Sql?,Ans : Windowing in streaming SQL involves defi...
433,data-engineering-zoomcamp,Triggers in Mage via CLI,"Encountering the error ""ModuleNotFoundError: N...","Python 3.12.1, is not compatible with kafka-py..."


Basics of Text Search:
1. Information Retrieval - The process of obtaining relevant information from large datasets based on user queries.
2. Vector Spaces - A mathematical representation where text is converted into vectors (points in space) allowing for quantitative comparison.
3. Bag of Words - A simple text representation model treating each document as a collection of words disregarding grammar and word order but keeping multiplicity.
4. TF-IDF (Term Frequency-Inverse Document Frequency) - A statistical measure used to evaluate how important a word is to a document in a collection or corpus. It increases with the number of times a word appears in the document but is offset by the frequency of the word in the corpus.


### What is Vector Spaces?

- turn the document into vector
- term-document matrix
    - row: documents ('Course - Can I still join the course after the start date?')
    - column: words/tokens
        - join is 1
        - course is 1
        - start is 1
        - date is 1
- in sklearn we have something called CountVectorizer(turning text into vectors)

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()
cv

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"
,analyzer,'word'


In [8]:
cv.fit(df.text)

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"
,analyzer,'word'


In [9]:
cv.get_feature_names_out()

array(['00', '00000000e', '0002', ..., '要了解键盘快捷键', '要启用屏幕阅读器支持', '请按ctrl'],
      shape=(6711,), dtype=object)

In [10]:
# There is a lot of noise here, so we will use only 5 documents.
cv = CountVectorizer(min_df=5)
cv.fit(df.text)
cv.get_feature_names_out()

array(['01', '02', '03', ..., 'youtube', 'zip', 'zoomcamp'],
      shape=(1524,), dtype=object)

In [11]:
docs_examples = [
    "Course starts on 15th Jan 2024",
    "Prerequisites listed on GitHub",
    "Submit homeworks after start date",
    "Registration not required for participation",
    "Setup Google Cloud and Python before course"
]

In [12]:
docs_examples

['Course starts on 15th Jan 2024',
 'Prerequisites listed on GitHub',
 'Submit homeworks after start date',
 'Registration not required for participation',
 'Setup Google Cloud and Python before course']

In [13]:
cv = CountVectorizer()
cv.fit(docs_examples)

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"
,analyzer,'word'


In [14]:
cv.get_feature_names_out() # all the words in the doc_examples

array(['15th', '2024', 'after', 'and', 'before', 'cloud', 'course',
       'date', 'for', 'github', 'google', 'homeworks', 'jan', 'listed',
       'not', 'on', 'participation', 'prerequisites', 'python',
       'registration', 'required', 'setup', 'start', 'starts', 'submit'],
      dtype=object)

In [15]:
X = cv.transform(docs_examples)
X.shape

(5, 25)

In [16]:
X

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 27 stored elements and shape (5, 25)>

In [17]:
X.todense()

matrix([[1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0,
         0, 0, 1, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0,
         0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 1, 0, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1,
         0, 0, 0, 0],
        [0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
         1, 0, 0, 0]])

In [18]:
# If we want to see what is inside
pd.DataFrame(X.todense(), columns=cv.get_feature_names_out())

,15th,2024,after,and,before,cloud,course,date,for,github,...,on,participation,prerequisites,python,registration,required,setup,start,starts,submit
0,1,1,0,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0,1,0
1,0,0,0,0,0,0,0,0,0,1,...,1,0,1,0,0,0,0,0,0,0
2,0,0,1,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,1,0,1
3,0,0,0,0,0,0,0,0,1,0,...,0,1,0,0,1,1,0,0,0,0
4,0,0,0,1,1,1,1,0,0,0,...,0,0,0,1,0,0,1,0,0,0


In [19]:
pd.DataFrame(X.todense(), columns=cv.get_feature_names_out()).T

,0,1,2,3,4
15th,1,0,0,0,0
2024,1,0,0,0,0
after,0,0,1,0,0
and,0,0,0,0,1
before,0,0,0,0,1
cloud,0,0,0,0,1
course,1,0,0,0,1
date,0,0,1,0,0
for,0,0,0,1,0
github,0,1,0,0,0


- turn the document into vector
- term-document matrix
    - row: documents ('Course - Can I still join the course after the start date?')
    - column: words/tokens
        - join is 1
        - course is 1
        - start is 1
        - date is 1
- in sklearn we have something called CountVectorizer(turning text into vectors)
- bag of words
    - order is not important in words.
    - sparse matrix.


In [20]:
# remove stop words like on, not, at, for, after, before
cv = CountVectorizer(stop_words='english')
cv.fit(docs_examples)

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,stop_words,'english'
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"
,analyzer,'word'


In [21]:
cv.get_feature_names_out()

array(['15th', '2024', 'cloud', 'course', 'date', 'github', 'google',
       'homeworks', 'jan', 'listed', 'participation', 'prerequisites',
       'python', 'registration', 'required', 'setup', 'start', 'starts',
       'submit'], dtype=object)

In [22]:
X = cv.transform(docs_examples)
pd.DataFrame(X.todense(), columns=cv.get_feature_names_out()).T

,0,1,2,3,4
15th,1,0,0,0,0
2024,1,0,0,0,0
cloud,0,0,0,0,1
course,1,0,0,0,1
date,0,0,1,0,0
github,0,1,0,0,0
google,0,0,0,0,1
homeworks,0,0,1,0,0
jan,1,0,0,0,0
listed,0,1,0,0,0


In [91]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words='english', min_df=5)
X = cv.fit_transform(df.text)

names = cv.get_feature_names_out()

df_docs = pd.DataFrame(X.toarray(), columns=names)
df_docs
# more than 900 documents and in the columns we can see the words

,01,02,03,04,05,06,09,10,100,11,...,y_val,yaml,year,yellow,yellow_tripdata_2021,yes,yml,youtube,zip,zoomcamp
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
943,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
944,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
945,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
946,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### TF-IDF (Term Frequency-Inverse Document Frequency) 

A statistical measure used to evaluate how important a word is to a document in a collection or corpus. It increases with the number of times a word appears in the document but is offset by the frequency of the word in the corpus.
Meaning

Yes — that sentence is describing **TF-IDF** (Term Frequency–Inverse Document Frequency), which is a common weighting method in text analysis.

Here’s the breakdown:

* **"It increases with the number of times a word appears in the document"** →
  That’s the **TF** part: *Term Frequency*. If a word shows up more often in a document, it gets a higher score.

* **"but is offset by the frequency of the word in the corpus"** →
  That’s the **IDF** part: *Inverse Document Frequency*. If a word appears in **many documents across the whole collection**, it becomes *less special*, so the score is reduced.

Putting it together:

* **Rare words** that appear frequently in a document get **high scores**.
* **Common words** like “yes,” “and,” “the” get **low scores** even if they appear a lot, because they don’t help distinguish documents.
* In our example: `"yes"` appears in many documents → low importance; `"yml"` appears rarely in the corpus → higher importance.



**Setup**

* Corpus size $N=5$ documents.
* Document $D$ has **100 words**.
* In $D$: `"yes"` appears **6** times; `"yml"` appears **2** times.
* Document frequencies: `"yes"` is in **4** of 5 docs; `"yml"` is in **1** of 5 docs.

**Formulas**

* $\text{TF}(t,D) = \frac{\text{count of } t \text{ in } D}{\text{total words in } D}$
* $\text{IDF}(t) = \ln\!\left(\frac{N}{\text{df}(t)}\right)$
* $\text{TF-IDF}(t,D) = \text{TF}(t,D)\times \text{IDF}(t)$

**Numbers**

* TF:

  * $\text{TF}(\text{"yes"},D)=6/100=0.06$
  * $\text{TF}(\text{"yml"},D)=2/100=0.02$
* IDF:

  * $\text{IDF}(\text{"yes"})=\ln(5/4)\approx 0.2231$
  * $\text{IDF}(\text{"yml"})=\ln(5/1)\approx 1.6094$
* TF-IDF:

  * $\text{TF-IDF}(\text{"yes"},D)=0.06\times 0.2231\approx \mathbf{0.0134}$
  * $\text{TF-IDF}(\text{"yml"},D)=0.02\times 1.6094\approx \mathbf{0.0322}$

**Takeaway**
Even though `"yes"` appears more often **in the document**, `"yml"` is much **rarer in the corpus**, so it gets a **\~2.4× higher** TF-IDF score. This is exactly the “more common ⇒ less important” intuition.


### Query-Document Similarity

In [92]:
from sklearn.feature_extraction.text import TfidfVectorizer

cv = TfidfVectorizer(stop_words='english', min_df=5)
X = cv.fit_transform(df.text)
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 23808 stored elements and shape (948, 1333)>

In [93]:
query = "Do I need to know python to sign up for the January course?"
q = cv.transform([query])
q.toarray()

array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 1333))

In [94]:
query_dict = dict(zip(names, q.toarray()[0]))
filtered = {k: v for k, v in query_dict.items() if v > 0}
filtered

{'course': np.float64(0.38148200594064524),
 'know': np.float64(0.5608269127690405),
 'need': np.float64(0.29796783250107517),
 'python': np.float64(0.31441356049301333),
 'sign': np.float64(0.5935519664108326)}

In [95]:
doc_dict = dict(zip(names, X.toarray()[2]))
filtered = {k: v for k, v in doc_dict.items() if v > 0}
filtered

{'don': np.float64(0.5310683382058037),
 'final': np.float64(0.38088037206388314),
 'homeworks': np.float64(0.38088037206388314),
 'projects': np.float64(0.2982703425530899),
 'register': np.float64(0.399850779948394),
 'submit': np.float64(0.31724075043760075),
 'yes': np.float64(0.2798913490945963)}

#### Dot Product

In [96]:
X.dot(q.T).todense() # cosine similiarity

matrix([[0.19464486],
        [0.        ],
        [0.        ],
        [0.06011641],
        [0.04932915],
        [0.        ],
        [0.        ],
        [0.13477565],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.15899187],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.07431408],
        [0.        ],
        [0.        ],
        [0.05779673],
        [0.07243428],
        [0.        ],
        [0.05174293],
        [0.16373635],
        [0.08076031],
        [0.        ],
        [0.09755254],
        [0.        ],
        [0.21069625],
        [0.12067781],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.06381749],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.        ],
        [0.00910541],
        [0.02835681],
        [0.05480112],
        [0.        ],
        [0.        ],
        [0.        ],
        [0

In [97]:
from sklearn.metrics.pairwise import cosine_similarity

In [98]:
score = cosine_similarity(X,q).flatten() # same result as the dot product

In [99]:
import numpy as np
np.argsort(score) # these are index sorted following the score started with zero for example the index
# 524 has zero score, and we're interested with the indexes at the end 27, 806, 577, 445

array([524, 800, 801, 807, 788, 790, 791, 792, 793, 794, 795, 796, 799,
       525, 526, 783, 784, 785, 786, 515, 516, 517, 518, 492, 498, 499,
       501, 504, 505, 487, 488, 489, 490, 491, 519, 493, 494, 479, 480,
       481, 482, 483, 486, 797, 798, 595, 809, 811, 812, 813, 527, 528,
       530, 585, 586, 590, 594, 808, 597, 576, 578, 579, 580, 581, 582,
       583, 584, 568, 546, 520, 523, 551, 552, 553, 555, 556, 558, 542,
       544, 497, 548, 549, 550, 532, 533, 534, 535, 536, 537, 538, 827,
       385, 832, 833, 834, 836, 839, 840, 822, 823, 824, 825, 384, 828,
       830, 430, 815, 816, 817, 818, 819, 820, 821, 389, 419, 400, 402,
       404, 405, 407, 408, 409, 386, 387, 420, 390, 392, 397, 399, 376,
       377, 379, 380, 382, 383, 437, 441, 442, 443, 444, 447, 453, 841,
       843, 846, 432, 468, 506, 507, 508, 509, 510, 512, 513, 514, 495,
       496, 474, 421, 422, 423, 426, 427, 428, 429, 471, 472, 473, 569,
       475, 476, 477, 478, 460, 461, 462, 463, 466, 467,  30, 70

In [100]:
np.argsort(score)[-5:]

array([764,  27, 806, 577, 445])

In [101]:
query = "I just discovered the course, is it too late to join?"
q = cv.transform([query])
q.toarray()


array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 1333))

In [102]:
score = cosine_similarity(X,q).flatten() 

In [103]:
np.argsort(score) [-5:]

array([ 22, 448, 449, 440,   0])

In [104]:
# these indexes represent documents
df.iloc[22].text

"It's up to you which platform and environment you use for the course.\nGithub codespaces or GCP VM are just possible options, but you can do the entire course from your laptop."

In [105]:
df.iloc[449].text

'Yes, you can. You won’t be able to submit some of the homeworks, but you can still take part in the course.\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers’ Projects by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'

In [106]:
df.iloc[0].text

"The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel."

### Vectorizing all the documents

In [107]:
fields = ['section', 'question', 'text']

In [108]:
matrices = {}
vectorizers = {}

for f in fields:
    cv = TfidfVectorizer(stop_words='english', min_df=5)
    X = cv.fit_transform(df[f])
    matrices[f] = X
    vectorizers[f] = cv

In [109]:
matrices

{'section': <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 3090 stored elements and shape (948, 66)>,
 'question': <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 3431 stored elements and shape (948, 291)>,
 'text': <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 23808 stored elements and shape (948, 1333)>}

section has 66 tokens, and text has the highest number of tokens which is 1333

In [110]:
vectorizers

{'section': TfidfVectorizer(min_df=5, stop_words='english'),
 'question': TfidfVectorizer(min_df=5, stop_words='english'),
 'text': TfidfVectorizer(min_df=5, stop_words='english')}

### Search


In [111]:
n = len(df)

In [113]:
score = np.zeros(n)
query = "I just discovered the course, is it too late to join?"

for f in fields:
    q = vectorizers[f].transform([query])
    X = matrices[f]

    f_score = cosine_similarity(X, q).flatten()
    score = score + f_score
    

In [122]:
filters = {
    'course': 'data-engineering-zoomcamp'
}

In [123]:
for field, value in filters.items():
    mask = (df[field] == value).astype(int).values
    score = score * mask

In [126]:
idx = np.argsort(score)[-11:]

In [127]:
df.iloc[idx]

,course,section,question,text
2,data-engineering-zoomcamp,General course-related questions,Course - Can I still join the course after the...,"Yes, even if you don't register, you're still ..."
11,data-engineering-zoomcamp,General course-related questions,Certificate - Can I follow the course in a sel...,"No, you can only get a certificate if you fini..."
10,data-engineering-zoomcamp,General course-related questions,Course - ​​How many hours per week am I expect...,It depends on your background and previous exp...
3,data-engineering-zoomcamp,General course-related questions,Course - I have registered for the Data Engine...,You don't need it. You're accepted. You can al...
1,data-engineering-zoomcamp,General course-related questions,Course - What are the prerequisites for this c...,GitHub - DataTalksClub data-engineering-zoomca...
4,data-engineering-zoomcamp,General course-related questions,Course - What can I do before the course starts?,You can start by installing and setting up all...
5,data-engineering-zoomcamp,General course-related questions,Course - how many Zoomcamps in a year?,"There are 3 Zoom Camps in a year, as of 2024. ..."
9,data-engineering-zoomcamp,General course-related questions,Course - Which playlist on YouTube should I re...,All the main videos are stored in the Main “DA...
34,data-engineering-zoomcamp,General course-related questions,How can we contribute to the course?,Star the repo! Share it with friends if you fi...
7,data-engineering-zoomcamp,General course-related questions,Course - Can I follow the course after it fini...,"Yes, we will keep all the materials after the ..."


### Search with all the fields & boosting + filtering
We can do it for all the fields. Let's also boost one of the fields - question - to give it more importance than to others

In [129]:
score = np.zeros(n)
query = "I just discovered the course, is it too late to join?"

boosts = {'question': 3}

for f in fields:
    q = vectorizers[f].transform([query])
    X = matrices[f]

    f_score = cosine_similarity(X, q).flatten()
    boost = boosts.get(f, 1.0)
    
    score = score + boost * f_score

In [130]:
filters = {
    'course': 'data-engineering-zoomcamp'
}

In [131]:
for field, value in filters.items():
    mask = (df[field] == value).astype(int).values
    score = score * mask

In [134]:
idx = np.argsort(-score)[:5]

In [135]:
df.iloc[idx]

,course,section,question,text
7,data-engineering-zoomcamp,General course-related questions,Course - Can I follow the course after it fini...,"Yes, we will keep all the materials after the ..."
0,data-engineering-zoomcamp,General course-related questions,Course - When will the course start?,The purpose of this document is to capture fre...
4,data-engineering-zoomcamp,General course-related questions,Course - What can I do before the course starts?,You can start by installing and setting up all...
1,data-engineering-zoomcamp,General course-related questions,Course - What are the prerequisites for this c...,GitHub - DataTalksClub data-engineering-zoomca...
5,data-engineering-zoomcamp,General course-related questions,Course - how many Zoomcamps in a year?,"There are 3 Zoom Camps in a year, as of 2024. ..."


### Putting it all together

In [136]:
class TextSearch:

    def __init__(self, text_fields):
        self.text_fields = text_fields
        self.matrices = {}
        self.vectorizers = {}

    def fit(self, records, vectorizer_params={}):
        self.df = pd.DataFrame(records)

        for f in self.text_fields:
            cv = TfidfVectorizer(**vectorizer_params)
            X = cv.fit_transform(self.df[f])
            self.matrices[f] = X
            self.vectorizers[f] = cv

    def search(self, query, n_results=10, boost={}, filters={}):
        score = np.zeros(len(self.df))

        for f in self.text_fields:
            b = boost.get(f, 1.0)
            q = self.vectorizers[f].transform([query])
            s = cosine_similarity(self.matrices[f], q).flatten()
            score = score + b * s

        for field, value in filters.items():
            mask = (self.df[field] == value).values
            score = score * mask

        idx = np.argsort(-score)[:n_results]
        results = self.df.iloc[idx]
        return results.to_dict(orient='records')

In [137]:
index = TextSearch(
    text_fields=['section', 'question', 'text']
)
index.fit(documents)

index.search(
    query='I just signed up. Is it too late to join the course?',
    n_results=5,
    boost={'question': 3.0},
    filters={'course': 'data-engineering-zoomcamp'}
)

[{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
  'section': 'General course-related questions',
  'question': 'Course - Can I still join the course after the start date?',
  'course': 'data-engineering-zoomcamp'},
 {'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
  'section': 'General course-related questions',
  'question': 'Course - When will the course start?',
  'course': 'data-engineerin

https://github.com/alexeygrigorev/minsearch

## 4. Embeddings and Vector Search

Problem with text - only exact matches. How about synonyms?
What are Embeddings?

* Conversion to Numbers: Embeddings transform different words, sentences and documents into dense vectors (arrays with numbers).
* Capturing Similarity: They ensure similar items have similar numerical vectors, illustrating their closeness in terms of characteristics.
* Dimensionality Reduction: Embeddings reduce complex characteristics into vectors.
* Use in Machine Learning: These numerical vectors are used in machine learning models for tasks such as recommendations, text analysis, and pattern recognition.


### SVD

Singular Value Decomposition is methond of dimensionality reduction and is the simplest way to turn Bag-of-Words representation into embeddings

This way we still don't preserve the word order (because it wasn't in the Bag-of-Words representation) but we reduce dimensionality and capture synonyms.

We won't go into mathematics, it's sufficient to know that SVD "compresses" our input vectors in such a way that as much as possible of the original information is retained.

This compression is lossy compression - meaning that we won't be able to restore the 100% of the original vector, but the result is close enough.

http://wordvec.colorado.edu/papers/Deerwester_1990.pdf

In [143]:
from sklearn.decomposition import TruncatedSVD

X = matrices['text']
cv = vectorizers['text']

In [144]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 23808 stored elements and shape (948, 1333)>

In [145]:
svd = TruncatedSVD(n_components=16)
X_emb = svd.fit_transform(X)

In [146]:
X_emb.shape # from 1333 to 16

(948, 16)

In [148]:
X_emb[0] # this is called embedding,SVD try to capture from the original as possible,
# and similar words will be grouped, vectores capture similiarty between words. And this for documents

array([ 0.0965348 , -0.08210778, -0.10316559, -0.07898142,  0.06771171,
       -0.05631715,  0.01265507, -0.12680151,  0.25960265,  0.28286648,
        0.02866466,  0.07173143, -0.12028062, -0.08976287,  0.02666054,
        0.00319301])

In [149]:
query = 'I just signed up. Is it too late to join the course?'

Q = cv.transform([query])
Q_emb = svd.transform(Q)
Q_emb[0]# represent the query

array([ 0.057904  , -0.03844236, -0.05671939, -0.02815313,  0.03949087,
       -0.06164733,  0.00867858, -0.08231631,  0.17631283,  0.18267787,
        0.02988303,  0.0692926 , -0.08384569, -0.04499617,  0.03041788,
       -0.01782537])

https://youtu.be/nMrGK5QgPVE?t=4321